# CoGAPS - Timing (3 runs)

💡 **Environment:** `clamp-analyses`  

In [ ]:
library(CoGAPS)
library(here)

In [ ]:
output_dir <- here("output/model_performance/gtex")
dir.create(output_dir, showWarnings = FALSE, recursive = TRUE)

N_RUNS <- 3

In [ ]:
gtex_data <- readRDS(here("output/gtex/df_gtex_fbm_filt.rds"))
K <- readRDS(here("output/gtex/CLAMP_K_gtex.rds"))
K <- as.integer(K)

gtex_data_shifted <- gtex_data - min(gtex_data)

In [ ]:
n_genes <- nrow(gtex_data_shifted)
nSets <- ceiling(n_genes / 2500)

params <- CogapsParams(
    nPatterns = K,
    nIterations = 5000,
    seed = 42,
    distributed = "genome-wide"
)
params <- setDistributedParams(params, nSets = nSets)

In [ ]:
CoGAPS_times <- numeric(N_RUNS)

for (i in 1:N_RUNS) {
  cat("CoGAPS run", i, "of", N_RUNS, "\n")
  
  start_time <- Sys.time()
  
  cogapsresult <- CoGAPS(gtex_data_shifted, params, nThreads = 4, outputFrequency = 10000)
  
  end_time <- Sys.time()
  CoGAPS_times[i] <- as.numeric(difftime(end_time, start_time, units = "mins"))
  cat("Run", i, "time:", CoGAPS_times[i], "minutes\n\n")
}

In [ ]:
CoGAPS_time_minutes <- CoGAPS_times
names(CoGAPS_time_minutes) <- paste0("run", 1:N_RUNS)
saveRDS(CoGAPS_time_minutes, file.path(output_dir, "CoGAPS_time_minutes.rds"))
cat("CoGAPS times:", CoGAPS_time_minutes, "minutes\n")